<a href="https://colab.research.google.com/github/Aleksei185/football-odds-lstm/blob/main/notebooks/01_lstm_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Загрузка данных из SQLite


In [5]:
from google.colab import drive
drive.mount('/content/drive')

import sqlite3
import pandas as pd
import numpy as np

db_path = '/content/drive/MyDrive/database.sqlite'
conn = sqlite3.connect(db_path)

print("\nЗагрузка таблицы Match (только Bet365)")
query_match = """
SELECT
    m.id AS match_id,
    m.date,
    m.season,
    m.stage,
    m.home_team_api_id,
    m.away_team_api_id,
    m.home_team_goal,
    m.away_team_goal,
    m.B365H, m.B365D, m.B365A
FROM Match m
WHERE m.date IS NOT NULL
"""
df_matches = pd.read_sql_query(query_match, conn)
print(f"Загружено {len(df_matches)} матчей")

print("\nЗагрузка таблицы Team...")
df_teams = pd.read_sql_query("SELECT team_api_id, team_long_name FROM Team", conn)
print(f"Загружено {len(df_teams)} команд")

# Загружаем таблицу Team_Attributes (БЕЗ buildUpPlayDribbling)
print("\nЗагрузка таблицы Team_Attributes...")
query_team_attr = """
SELECT
    team_api_id,
    date,
    buildUpPlaySpeed,
    buildUpPlayPassing,
    chanceCreationPassing,
    chanceCreationCrossing,
    chanceCreationShooting,
    defencePressure,
    defenceAggression,
    defenceTeamWidth
FROM Team_Attributes
"""
df_team_attr = pd.read_sql_query(query_team_attr, conn)
df_team_attr['date'] = pd.to_datetime(df_team_attr['date'])
print(f"Загружено {len(df_team_attr)} записей об атрибутах")

# Объединяем матчи с названиями команд
print("\nОбъединение с названиями команд")
df_matches['date'] = pd.to_datetime(df_matches['date'])

df_matches = df_matches.merge(
    df_teams, left_on='home_team_api_id', right_on='team_api_id', how='left'
).rename(columns={'team_long_name': 'home_team_name'}).drop('team_api_id', axis=1)

df_matches = df_matches.merge(
    df_teams, left_on='away_team_api_id', right_on='team_api_id', how='left'
).rename(columns={'team_long_name': 'away_team_name'}).drop('team_api_id', axis=1)
print("Названия команд добавлены")

# Объединение с атрибутами через merge_asof
print("\nОбъединение с атрибутами команд (оптимизированно)")

df_matches = df_matches.sort_values('date')
df_team_attr = df_team_attr.sort_values('date')

def merge_attributes_fast(matches, attributes, team_col, prefix):
    attrs = attributes.rename(columns={'team_api_id': team_col})
    matches_sorted = matches.sort_values('date')
    attrs_sorted = attrs.sort_values('date')

    merged = pd.merge_asof(
        matches_sorted,
        attrs_sorted,
        on='date',
        by=team_col,
        direction='nearest'
    )

    attr_cols = ['buildUpPlaySpeed', 'buildUpPlayPassing',
                 'chanceCreationPassing', 'chanceCreationCrossing', 'chanceCreationShooting',
                 'defencePressure', 'defenceAggression', 'defenceTeamWidth']

    for col in attr_cols:
        merged.rename(columns={col: f'{prefix}_{col}'}, inplace=True)

    return merged

df_matches = merge_attributes_fast(df_matches, df_team_attr, 'home_team_api_id', 'home')
df_matches = merge_attributes_fast(df_matches, df_team_attr, 'away_team_api_id', 'away')

print("Атрибуты команд добавлены")

# 7. Удаляем строки с пропусками (ТОЛЬКО Bet365 + атрибуты)
print("\nУдаление строк с пропусками...")

# Список колонок, где не должно быть пропусков
cols_to_check = ['B365H', 'B365D', 'B365A'] + [col for col in df_matches.columns if col.startswith('home_') or col.startswith('away_')]

df_matches = df_matches.dropna(subset=cols_to_check)
print(f"Осталось {len(df_matches)} матчей после удаления пропусков")

# 8. Выводим итоговую информацию
print("\n" + "="*60)
print("ИТОГОВАЯ ИНФОРМАЦИЯ О ДАННЫХ")
print("="*60)
print(f"Размер датасета: {df_matches.shape[0]} строк, {df_matches.shape[1]} колонок")
print(f"\nСписок колонок ({len(df_matches.columns)}):")
print(df_matches.columns.tolist())
print(f"\nПервые 3 строки:")
display(df_matches.head(3))
print(f"\nПропуски в данных:")
print(df_matches.isnull().sum().sum(), "пропусков")

conn.close()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Загрузка таблицы Match (только Bet365)
Загружено 25979 матчей

Загрузка таблицы Team...
Загружено 299 команд

Загрузка таблицы Team_Attributes...
Загружено 1458 записей об атрибутах

Объединение с названиями команд
Названия команд добавлены

Объединение с атрибутами команд (оптимизированно)
Атрибуты команд добавлены

Удаление строк с пропусками...
Осталось 22309 матчей после удаления пропусков

ИТОГОВАЯ ИНФОРМАЦИЯ О ДАННЫХ
Размер датасета: 22309 строк, 29 колонок

Список колонок (29):
['match_id', 'date', 'season', 'stage', 'home_team_api_id', 'away_team_api_id', 'home_team_goal', 'away_team_goal', 'B365H', 'B365D', 'B365A', 'home_team_name', 'away_team_name', 'home_buildUpPlaySpeed', 'home_buildUpPlayPassing', 'home_chanceCreationPassing', 'home_chanceCreationCrossing', 'home_chanceCreationShooting', 'home_defencePressure', 'home_defenceAggression', 'home_d

,match_id,date,season,stage,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,B365H,B365D,...,home_defenceAggression,home_defenceTeamWidth,away_buildUpPlaySpeed,away_buildUpPlayPassing,away_chanceCreationPassing,away_chanceCreationCrossing,away_chanceCreationShooting,away_defencePressure,away_defenceAggression,away_defenceTeamWidth
22,4769,2008-08-09,2008/2009,1,8583,9830,2,1,2.10,3.1,...,55.0,30.0,55.0,35.0,60.0,65.0,35.0,30.0,45.0,30.0
23,19694,2008-08-09,2008/2009,1,8596,8548,0,1,6.50,4.0,...,70.0,70.0,65.0,50.0,70.0,70.0,70.0,60.0,70.0,70.0
24,4775,2008-08-09,2008/2009,1,8481,8639,0,0,2.15,3.1,...,70.0,70.0,70.0,60.0,50.0,45.0,55.0,55.0,70.0,55.0



Пропуски в данных:
0 пропусков
